#### Note: All data in this repo is synthetically made including sleep stage annotations, demographics or diseases. The data is for demo purposes only.


In [1]:
%load_ext autoreload
%autoreload 2

#### Preprocessing Details


Before running this notebook, please preprocess your PSG files using the scripts provided in `sleepfm/preprocessing`. Note that PSG recordings may contain different sets of channels across datasets. The predefined channel–modality mappings used in this project are specified in `sleepfm/configs/channel_groups.json`.

Although we have attempted to make this mapping as comprehensive as possible, we strongly recommend reviewing the channels present in your specific PSG data. In consultation with domain experts, you should group any additional or dataset-specific channels into the appropriate modality categories and update `channel_groups.json` accordingly. This step is critical to ensure that all channels are correctly aligned with their intended modalities during preprocessing and downstream modeling.

In [ ]:
import torch
from torch import nn
import numpy as np
from sklearn.metrics import f1_score, confusion_matrix
import os
import tqdm
import random
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from collections import Counter
import pandas as pd
sys.path.append("..")
sys.path.append("../sleepfm")
from preprocessing.preprocessing import EDFToHDF5Converter
from models.dataset import SetTransformerDataset, collate_fn
from models.models import SetTransformer, SleepEventLSTMClassifier, DiagnosisFinetuneFullLSTMCOXPHWithDemo
import h5py
from utils import load_config, load_data, save_data, count_parameters
from torch.utils.data import Dataset, DataLoader

#### Part 0: Preprocessing EDF files

Note: This is just a demo notebook that preprocesses a single, specific file. run `sleepfm/preprocessing/preprocessing.sh` with appropriate folders to generate multiple preprocessed files.

In [ ]:
base_save_path = "demo_data"
os.makedirs(base_save_path, exist_ok=True)

In [ ]:
root_dir = "/edf_root"      # dummy root not used for a single file conversion
target_dir = "/note"    # dummy target not used for a single file conversion

edf_path = "demo_data/demo_psg.edf"
hdf5_path = os.path.join(base_save_path, "demo_psg.hdf5")

converter = EDFToHDF5Converter(
    root_dir=root_dir,
    target_dir=target_dir,
    resample_rate=128
)

# run for single file conversion
converter.convert(edf_path, hdf5_path)

#### Part 1: Generating embeddings from SleepFM pretrained model

Here we show generating embedding for 1 demno PSG. To see full script, please check `sleepfm/pipeline/generate_embeddings.py`. 

In [ ]:
model_path = "../sleepfm/checkpoints/model_base"
channel_groups_path = "../sleepfm/configs/channel_groups.json"
config_path = os.path.join(model_path, "config.json")

config = load_config(config_path)
channel_groups = load_data(channel_groups_path)

In [ ]:
modality_types = config["modality_types"]
in_channels = config["in_channels"]
patch_size = config["patch_size"]
embed_dim = config["embed_dim"]
num_heads = config["num_heads"]
num_layers = config["num_layers"]
pooling_head = config["pooling_head"]
dropout = 0.0

In [ ]:
model_class = getattr(sys.modules[__name__], config['model'])
model = model_class(in_channels, patch_size, embed_dim, num_heads, num_layers, pooling_head=pooling_head, dropout=dropout)

device = torch.device("cuda")
if device.type == "cuda":
    model = torch.nn.DataParallel(model)

model.to(device)
total_layers, total_params = count_parameters(model)
print(f'Trainable parameters: {total_params / 1e6:.2f} million')
print(f'Number of layers: {total_layers}')

In [ ]:
checkpoint = torch.load(os.path.join(model_path, "best.pt"))
model.load_state_dict(checkpoint["state_dict"])
model.eval()

In [ ]:
config

In [ ]:
hdf5_paths = [os.path.join(base_save_path, "demo_psg.hdf5")]
dataset = SetTransformerDataset(config, channel_groups, hdf5_paths=hdf5_paths, split="test")

dataloader = torch.utils.data.DataLoader(dataset, 
                                            batch_size=16, 
                                            num_workers=1, 
                                            shuffle=False, 
                                            collate_fn=collate_fn)

In [ ]:
output = os.path.join(base_save_path, "demo_emb")
output_5min_agg = os.path.join(base_save_path, "demo_5min_agg_emb")
os.makedirs(output, exist_ok=True)
os.makedirs(output_5min_agg, exist_ok=True)

In [ ]:
with torch.no_grad():
    with tqdm.tqdm(total=len(dataloader)) as pbar:
        for batch in dataloader:
            batch_data, mask_list, file_paths, dset_names_list, chunk_starts = batch
            (bas, resp, ekg, emg) = batch_data
            (mask_bas, mask_resp, mask_ekg, mask_emg) = mask_list

            bas = bas.to(device, dtype=torch.float)
            resp = resp.to(device, dtype=torch.float)
            ekg = ekg.to(device, dtype=torch.float)
            emg = emg.to(device, dtype=torch.float)

            mask_bas = mask_bas.to(device, dtype=torch.bool)
            mask_resp = mask_resp.to(device, dtype=torch.bool)
            mask_ekg = mask_ekg.to(device, dtype=torch.bool)
            mask_emg = mask_emg.to(device, dtype=torch.bool)

            embeddings = [
                model(bas, mask_bas),
                model(resp, mask_resp),
                model(ekg, mask_ekg),
                model(emg, mask_emg),
            ]

            # Model gives two kinds of embeddings. Granular 5 second-level embeddings and aggregated 5 minute-level embeddings. We save both of them below. 

            embeddings_new = [e[0].unsqueeze(1) for e in embeddings]

            for i in range(len(file_paths)):
                file_path = file_paths[i]
                chunk_start = chunk_starts[i]
                subject_id = os.path.basename(file_path).split('.')[0]
                output_path = os.path.join(output_5min_agg, f"{subject_id}.hdf5")

                with h5py.File(output_path, 'a') as hdf5_file:
                    for modality_idx, modality_type in enumerate(config["modality_types"]):
                        if modality_type in hdf5_file:
                            dset = hdf5_file[modality_type]
                            chunk_start_correct = chunk_start // (embed_dim * 5 * 60)
                            chunk_end = chunk_start_correct + embeddings_new[modality_idx][i].shape[0]
                            if dset.shape[0] < chunk_end:
                                dset.resize((chunk_end,) + embeddings_new[modality_idx][i].shape[1:])
                            dset[chunk_start_correct:chunk_end] = embeddings_new[modality_idx][i].cpu().numpy()
                        else:
                            hdf5_file.create_dataset(modality_type, data=embeddings_new[modality_idx][i].cpu().numpy(), chunks=(embed_dim,) + embeddings_new[modality_idx][i].shape[1:], maxshape=(None,) + embeddings_new[modality_idx][i].shape[1:])

            embeddings_new = [e[1] for e in embeddings]

            for i in range(len(file_paths)):
                file_path = file_paths[i]
                chunk_start = chunk_starts[i]
                subject_id = os.path.basename(file_path).split('.')[0]
                output_path = os.path.join(output, f"{subject_id}.hdf5")

                with h5py.File(output_path, 'a') as hdf5_file:
                    for modality_idx, modality_type in enumerate(config["modality_types"]):
                        if modality_type in hdf5_file:
                            dset = hdf5_file[modality_type]
                            chunk_start_correct = chunk_start // (embed_dim * 5)
                            chunk_end = chunk_start_correct + embeddings_new[modality_idx][i].shape[0]
                            if dset.shape[0] < chunk_end:
                                dset.resize((chunk_end,) + embeddings_new[modality_idx][i].shape[1:])
                            dset[chunk_start_correct:chunk_end] = embeddings_new[modality_idx][i].cpu().numpy()
                        else:
                            hdf5_file.create_dataset(modality_type, data=embeddings_new[modality_idx][i].cpu().numpy(), chunks=(embed_dim,) + embeddings_new[modality_idx][i].shape[1:], maxshape=(None,) + embeddings_new[modality_idx][i].shape[1:])
            pbar.update()

#### Part 2: Sleep Staging

Note that below, we are using our finetuned sleep staging model. It is always a good idea to finetune our model on your specific data, even if you only have a handful of sample, so that the model can adapt to your specific data distribution. Script to finetune your sleep staging model head is given in `sleepfm/pipeline/finetune_sleep_staging.py`. 

In [ ]:
sleep_staging_model_path = "../sleepfm/checkpoints/model_sleep_staging"
sleep_staging_config = load_data(os.path.join(sleep_staging_model_path, "config.json"))

sleep_staging_model_params = sleep_staging_config['model_params']
sleep_staging_model_class = getattr(sys.modules[__name__], sleep_staging_config['model'])

sleep_staging_model = sleep_staging_model_class(**sleep_staging_model_params).to(device)
sleep_staging_model_name = type(sleep_staging_model).__name__

In [ ]:
sleep_staging_model = nn.DataParallel(sleep_staging_model)
print(f"Using {torch.cuda.device_count()} GPUs")

In [ ]:
print(f"Model initialized: {sleep_staging_model_name}")
total_layers, total_params = count_parameters(sleep_staging_model)
print(f'Trainable parameters: {total_params / 1e6:.2f} million')
print(f'Number of layers: {total_layers}')

In [ ]:
sleep_staging_checkpoint_path = os.path.join(sleep_staging_model_path, "best.pth")
sleep_staging_checkpoint = torch.load(sleep_staging_checkpoint_path)
sleep_staging_model.load_state_dict(sleep_staging_checkpoint)

Below are some helper functions for loading data for sleep staging. You can find similar functions within `sleepfm/models/dataset.py`. You may need to modify it slightly based on your usecase. 

In [ ]:
class SleepEventClassificationDataset(Dataset):
    def __init__(
        self,
        config,
        channel_groups,
        hdf5_paths,
        label_files,
        split="train",
    ):
        self.config = config
        self.max_channels = self.config["max_channels"]
        self.context = int(self.config["context"])
        self.channel_like = self.config["channel_like"]

        self.max_seq_len = config["model_params"]["max_seq_length"]

        # --- Build label lookup: {study_id: label_csv_path} ---
        # study_id = filename without extension, e.g. "SSC_12345"
        labels_dict = {
            os.path.basename(p).rsplit(".", 1)[0]: p
            for p in label_files
            if os.path.exists(p)
        }

        # --- Filter to HDF5s that exist and have a matching label file ---
        hdf5_paths = [p for p in hdf5_paths if os.path.exists(p)]
        hdf5_paths = [
            p for p in hdf5_paths
            if os.path.basename(p).rsplit(".", 1)[0] in labels_dict
        ]

        if config.get("max_files"):
            hdf5_paths = hdf5_paths[: config["max_files"]]

        self.hdf5_paths = hdf5_paths
        self.labels_dict = labels_dict

        # --- Build index map ---
        # Each item is (hdf5_path, label_path, start_index)
        if self.context == -1:
            self.index_map = [
                (p, labels_dict[os.path.basename(p).rsplit(".", 1)[0]], -1)
                for p in self.hdf5_paths
            ]
        else:
            self.index_map = []
            loop = tqdm(self.hdf5_paths, total=len(self.hdf5_paths), desc=f"Indexing {split} data")
            for hdf5_file_path in loop:
                file_prefix = os.path.basename(hdf5_file_path).rsplit(".", 1)[0]
                label_path = labels_dict[file_prefix]

                with h5py.File(hdf5_file_path, "r") as hf:
                    dset_names = list(hf.keys())
                    if len(dset_names) == 0:
                        continue

                    # Use first dataset to define length (same as your original behavior)
                    first_name = dset_names[0]
                    dataset_length = hf[first_name].shape[0]

                for i in range(0, dataset_length, self.context):
                    self.index_map.append((hdf5_file_path, label_path, i))

        # If you have logger, keep; otherwise you can remove these.
        # logger.info(f"Number of files in {split} set: {len(self.hdf5_paths)}")
        # logger.info(f"Number of files to be processed in {split} set: {len(self.index_map)}")

        self.total_len = len(self.index_map)

    def __len__(self):
        return self.total_len

    def get_index_map(self):
        return self.index_map

    def __getitem__(self, idx):
        hdf5_path, label_path, start_index = self.index_map[idx]

        labels_df = pd.read_csv(label_path)
        labels_df["StageNumber"] = labels_df["StageNumber"].replace(-1, 0)

        y_data = labels_df["StageNumber"].to_numpy()
        if self.context != -1:
            y_data = y_data[start_index : start_index + self.context]

        x_data = []
        with h5py.File(hdf5_path, "r") as hf:
            dset_names = list(hf.keys())

            for dataset_name in dset_names:
                if dataset_name in self.channel_like:
                    if self.context == -1:
                        x_data.append(hf[dataset_name][:])
                    else:
                        x_data.append(hf[dataset_name][start_index : start_index + self.context])

        if not x_data:
            # Skip this data point if x_data is empty
            return self.__getitem__((idx + 1) % self.total_len)

        x_data = np.array(x_data)  # (C, T, F) assuming each channel returns (T, F)
        x_data = torch.tensor(x_data, dtype=torch.float32)
        y_data = torch.tensor(y_data, dtype=torch.float32)

        min_length = min(x_data.shape[1], len(y_data))
        x_data = x_data[:, :min_length, :]
        y_data = y_data[:min_length]

        return x_data, y_data, self.max_channels, self.max_seq_len, hdf5_path


def sleep_event_finetune_full_collate_fn(batch):
    x_data, y_data, max_channels_list, max_seq_len_list, hdf5_path_list = zip(*batch)

    num_channels = max(max_channels_list)

    max_seq_len_temp = max([item.size(1) for item in x_data])
    # Determine the max sequence length for padding
    if max_seq_len_list[0] is None:
        max_seq_len = max_seq_len_temp
    else:
        max_seq_len = min(max_seq_len_temp, max_seq_len_list[0])

    padded_x_data = []
    padded_y_data = []
    padded_mask = []

    for x_item, y_item in zip(x_data, y_data):

        # first non-zero index of y_data
        #print(y_item.shape)


        tgt_sleep_no_sleep = np.where(y_item > 0, 1, 0)
        moving_avg_tgt_sleep_no_sleep = np.convolve(tgt_sleep_no_sleep, np.ones(1080)/1080, mode='valid')
        try:
            first_non_zero_index = np.where(moving_avg_tgt_sleep_no_sleep > 0.5)[0][0]
        except IndexError:
            first_non_zero_index = 0



        #non_zero_indices = (y_item != 0).nonzero(as_tuple=True)[0]
        #first_non_zero_index = non_zero_indices[0].item() - 20
        if first_non_zero_index < 0:
            first_non_zero_index = 0

        #first_non_zero_index = 0

        #print(f"First non-zero index of y_data: {first_non_zero_index}")
        # Get the shape of x_item
        c, s, e = x_item.size()
        c = min(c, num_channels)
        s = min(s, max_seq_len + first_non_zero_index)  # Ensure the sequence length doesn't exceed max_seq_len

        # Create a padded tensor and a mask tensor for x_data
        padded_x_item = torch.zeros((num_channels, max_seq_len, e))
        mask = torch.ones((num_channels, max_seq_len))

        # Copy the actual data to the padded tensor and set the mask for real data
        #print(f"Shape of x_item: {x_item[:c, first_non_zero_index:s, :e].shape}")
        padded_x_item[:c, :s-first_non_zero_index, :e] = x_item[:c, first_non_zero_index:s, :e]
        mask[:c, :s-first_non_zero_index] = 0  # 0 for real data, 1 for padding

        # Pad y_data with zeros to match max_seq_len
        padded_y_item = torch.zeros(max_seq_len)
        padded_y_item[:s-first_non_zero_index] = y_item[first_non_zero_index:s]

        # Append padded items to lists
        padded_x_data.append(padded_x_item)
        padded_y_data.append(padded_y_item)
        padded_mask.append(mask)

    # Stack all tensors into a batch
    x_data = torch.stack(padded_x_data)
    y_data = torch.stack(padded_y_data)
    padded_mask = torch.stack(padded_mask)

    '''
    for y_data_mini in y_data:
        unique_labels = torch.unique(y_data_mini)
        print(f"Unique labels in batch: {unique_labels}")
    '''

    return x_data, y_data, padded_mask, hdf5_path_list

In [ ]:
hdf5_paths = [os.path.join(base_save_path, "demo_emb/demo_psg.hdf5")]
label_files = [os.path.join(base_save_path, "demo_psg.csv")]
test_dataset = SleepEventClassificationDataset(sleep_staging_config, channel_groups, split="test", hdf5_paths=hdf5_paths, label_files=label_files)

In [ ]:
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, num_workers=1, collate_fn=sleep_event_finetune_full_collate_fn)

In [ ]:
# Validation loop at the end of each epoch
model.eval()
all_targets = []
all_logits = []
all_outputs = []
all_masks = []
all_paths = []

count = 0
with torch.no_grad():
    for (x_data, y_data, padded_matrix, hdf5_path_list) in tqdm.tqdm(test_loader, desc="Evaluating"):
        x_data, y_data, padded_matrix, hdf5_path_list = x_data.to(device), y_data.to(device), padded_matrix.to(device), list(hdf5_path_list)
        outputs, mask = sleep_staging_model(x_data, padded_matrix)
        all_targets.append(y_data.cpu().numpy())
        all_outputs.append(torch.softmax(outputs, dim=-1).cpu().numpy())
        all_logits.append(outputs.cpu().numpy())
        all_masks.append(mask.cpu().numpy())
        all_paths.append(hdf5_path_list)


save_path = os.path.join(base_save_path, "demo_sleep_staging")
os.makedirs(save_path, exist_ok=True)

targets_path = os.path.join(save_path, "all_targets.pickle")
outputs_path = os.path.join(save_path, "all_outputs.pickle")
logits_path = os.path.join(save_path, "all_logits.pickle")
mask_path = os.path.join(save_path, "all_masks.pickle")
file_paths = os.path.join(save_path, "all_paths.pickle")

save_data(all_targets, targets_path)
save_data(all_outputs, outputs_path)
save_data(all_logits, logits_path)
save_data(all_masks, mask_path)
save_data(all_paths, file_paths)

In [ ]:
all_outputs[0].shape, all_targets[0].shape

In [ ]:
len(all_logits), len(all_outputs), len(all_targets), len(all_masks)

In [ ]:
all_logits[0].shape, all_outputs[0].shape, all_targets[0].shape, all_masks[0].shape

In [ ]:
all_logits_flat = [logits.reshape(-1, logits.shape[-1]) for logits in all_logits]
all_outputs_flat = [outputs.reshape(-1, outputs.shape[-1]) for outputs in all_outputs]
all_targets_flat = [targets.reshape(-1) for targets in all_targets]
all_masks_flat = [mask.reshape(-1) for mask in all_masks]

# Convert lists of flattened arrays to single concatenated arrays if desired
all_logits_flat = np.concatenate(all_logits_flat, axis=0)
all_outputs_flat = np.concatenate(all_outputs_flat, axis=0)
all_targets_flat = np.concatenate(all_targets_flat, axis=0)
all_masks_flat = np.concatenate(all_masks_flat, axis=0)

In [ ]:
all_logits_flat.shape, all_outputs_flat.shape, all_targets_flat.shape, all_masks_flat.shape

In [ ]:
mask_filter = all_masks_flat == 0

# Apply the mask to each flattened array
all_logits_filtered = all_logits_flat[mask_filter]
all_outputs_filtered = all_outputs_flat[mask_filter]
all_targets_filtered = all_targets_flat[mask_filter]

In [ ]:
counts = Counter(all_targets_filtered)
total = sum(counts.values())
prevalence_dict = {cls: count / total for cls, count in counts.items()}
prevalence_dict

In [ ]:
class_labels = ["Wake", "Stage 1", "Stage 2", "Stage 3", "REM"]
# class_labels = ["No-Apnea", "Apnea"]
class_mapping = {label: idx for idx, label in enumerate(class_labels)}

In [ ]:
# Step 1: Get predicted labels (argmax on probabilities)
predicted_labels = np.argmax(all_outputs_filtered, axis=1)

fontsize = 12

# Step 2: Compute F1 score for each class
f1_scores = f1_score(all_targets_filtered, predicted_labels, average=None, labels=range(len(class_labels)))
for idx, label in enumerate(class_labels):
    print(f"F1 Score for {label}: {f1_scores[idx]:.3f}")

# Step 3: Create a confusion matrix and normalize it by row to get percentages
conf_matrix = confusion_matrix(all_targets_filtered, predicted_labels, labels=range(len(class_labels)))
conf_matrix_percent = conf_matrix / conf_matrix.sum(axis=1, keepdims=True) * 100

# Plotting the confusion matrix with percentages
plt.figure(figsize=(6, 4))
sns.heatmap(
    conf_matrix_percent,
    annot=True,
    fmt=".1f",
    cmap="Blues",
    xticklabels=class_labels,
    yticklabels=class_labels,
    annot_kws={"size": fontsize},  # Font size for numbers inside the heatmap
    cbar_kws={"shrink": 1},  # Adjust colorbar size
)

# Customizing axis labels and ticks
plt.xlabel("Predicted Labels", fontsize=fontsize)
plt.ylabel("True Labels", fontsize=fontsize)
plt.xticks(fontsize=12, ha="center")  # Font size for x-axis tick labels with rotation
plt.yticks(fontsize=12)  # Font size for y-axis tick labels

# Adjust layout and save the figure
plt.tight_layout()
plt.show()

#### Part 3: Disease Prediction

In [ ]:
disease_model_path = "../sleepfm/checkpoints/model_diagnosis"
config = load_data(os.path.join(disease_model_path, "config.json"))

In [ ]:
config["model_params"]["dropout"] = 0.0
model_params = config['model_params']
model_class = getattr(sys.modules[__name__], config['model'])
model = model_class(**model_params).to(device)
model_name = type(model).__name__

In [ ]:
model = nn.DataParallel(model)
print(f"Model initialized: {model_name}")
total_layers, total_params = count_parameters(model)
print(f'Trainable parameters: {total_params / 1e6:.2f} million')
print(f'Number of layers: {total_layers}')

In [ ]:
checkpoint_path = os.path.join(disease_model_path, "best.pth")

In [ ]:
checkpoint = torch.load(checkpoint_path)
model.load_state_dict(checkpoint)

In [ ]:
class DiagnosisFinetuneFullCOXPHWithDemoDataset(Dataset):
    def __init__(self, 
                 config,
                 channel_groups,
                 hdf5_paths=None,
                 demo_labels_path=None,
                 split="train"):

        self.config = config
        self.channel_groups = channel_groups
        self.max_channels = self.config["max_channels"]

        # --- Load demographic features ---
        if not demo_labels_path:
            demo_labels_path = config["demo_labels_path"]

        demo_labels_df = pd.read_csv(demo_labels_path)
        demo_labels_df = demo_labels_df.set_index("Study ID")
        study_ids = set(demo_labels_df.index)

        is_event_df = pd.read_csv(os.path.join(self.config["labels_path"], "is_event.csv"))
        event_time_df = pd.read_csv(os.path.join(self.config["labels_path"], "time_to_event.csv"))

        is_event_df = is_event_df.set_index('Study ID')
        event_time_df = event_time_df.set_index('Study ID')

        # --- Resolve HDF5 paths (explicit precedence) ---
        if hdf5_paths:
            # Use provided paths directly
            hdf5_paths = [f for f in hdf5_paths if os.path.exists(f)]
        else:
            # Load from split file
            split_paths = load_data(config["split_path"])[split]
            hdf5_paths = [f for f in split_paths if os.path.exists(f)]

        # Filter by available demo labels
        hdf5_paths = [
            f for f in hdf5_paths
            if os.path.basename(f).split(".")[0] in study_ids
        ]

        # Optional truncation
        if config.get("max_files"):
            hdf5_paths = hdf5_paths[:config["max_files"]]

        labels_dict = {}
        # Loop over each study_id
        for study_id in tqdm.tqdm(study_ids):
            # Extract the row as a whole for both dataframes (faster than iterating over columns)
            is_event_row = list(is_event_df.loc[study_id].values)
            event_time_row = list(event_time_df.loc[study_id].values)
            demo_feats = list(demo_labels_df.loc[study_id].values)

            # values = [[event_time, is_event] for is_event, event_time in zip(is_event_row, event_time_row)]
            labels_dict[study_id] = {
                "is_event": is_event_row,
                "event_time": event_time_row, 
                "demo_feats": demo_feats
            }

        # --- Build index map ---
        self.index_map = [
            (path, labels_dict[os.path.basename(path).split(".")[0]])
            for path in hdf5_paths
        ]

        print(f"Number of files in {split} set: {len(hdf5_paths)}")
        print(f"Number of files to be processed in {split} set: {len(self.index_map)}")

        self.total_len = len(self.index_map)
        self.max_seq_len = config["model_params"]["max_seq_length"]

        if self.total_len == 0:
            raise ValueError(f"No valid HDF5 files found for split='{split}'.")

    def __len__(self):
        return self.total_len

    def __getitem__(self, idx):
        hdf5_path, tte_event = self.index_map[idx]

        event_time = tte_event["event_time"]
        is_event = tte_event["is_event"]
        demo_feats = tte_event["demo_feats"]

        x_data = []
        with h5py.File(hdf5_path, 'r') as hf:
            dset_names = []
            for dset_name in hf.keys():
                if isinstance(hf[dset_name], h5py.Dataset) and dset_name in self.config["modality_types"]:
                    dset_names.append(dset_name)
            
            random.shuffle(dset_names)
            for dataset_name in dset_names:
                x_data.append(hf[dataset_name][:])

        if not x_data:
            # Skip this data point if x_data is empty
            return self.__getitem__((idx + 1) % self.total_len)

        # Convert x_data list to a single numpy array
        x_data = np.array(x_data)

        # Convert x_data to tensor
        x_data = torch.tensor(x_data, dtype=torch.float32)

        event_time = torch.tensor(event_time, dtype=torch.float32)
        is_event = torch.tensor(is_event) 

        demo_feats = torch.tensor(demo_feats, dtype=torch.float32)

        return x_data, event_time, is_event, demo_feats, self.max_channels, self.max_seq_len, hdf5_path


def diagnosis_finetune_full_coxph_with_demo_collate_fn(batch):
    x_data, event_time, is_event, demo_feats, max_channels_list, max_seq_len_list, hdf5_path_list = zip(*batch)

    num_channels = max(max_channels_list)

    if max_seq_len_list[0] == None:
        max_seq_len = max([item.size(1) for item in x_data])
    else:
        max_seq_len = max_seq_len_list[0]

    padded_x_data = []
    padded_mask = []
    for item in x_data:
        c, s, e = item.size()
        c = min(c, num_channels)
        s = min(s, max_seq_len)  # Ensure the sequence length doesn't exceed max_seq_len

        # Create a padded tensor and a mask tensor
        padded_item = torch.zeros((num_channels, max_seq_len, e))
        mask = torch.ones((num_channels, max_seq_len))

        # Copy the actual data to the padded tensor and set the mask for real data
        padded_item[:c, :s, :e] = item[:c, :s, :e]
        mask[:c, :s] = 0  # 0 for real data, 1 for padding

        padded_x_data.append(padded_item)
        padded_mask.append(mask)
    
    # Stack all tensors into a batch
    x_data = torch.stack(padded_x_data)
    event_time = torch.stack(event_time)
    is_event = torch.stack(is_event)
    demo_feats = torch.stack(demo_feats)
    padded_mask = torch.stack(padded_mask)
    
    return x_data, event_time, is_event, demo_feats, padded_mask, hdf5_path_list

In [ ]:
save_path = os.path.join(base_save_path, "demo_diagnosis")
os.makedirs(save_path, exist_ok=True)

In [ ]:
hdf5_paths = [os.path.join(base_save_path, "demo_emb/demo_psg.hdf5")]
demo_labels_path = os.path.join(base_save_path, "demo_age_gender.csv")
config["labels_path"] = base_save_path

test_dataset = DiagnosisFinetuneFullCOXPHWithDemoDataset(config, channel_groups, split="test", hdf5_paths=hdf5_paths, demo_labels_path=demo_labels_path)

In [ ]:
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, num_workers=1, collate_fn=diagnosis_finetune_full_coxph_with_demo_collate_fn)

In [ ]:
model.eval()
all_event_times = []
all_is_event = []
all_outputs = []
all_paths = []

with torch.no_grad():
    for item in tqdm.tqdm(test_loader, desc="Evaluating"):
        x_data, event_times, is_event, demo_feats, padded_matrix, hdf5_path_list = item
        x_data, event_times, is_event, demo_feats, padded_matrix, hdf5_path_list = x_data.to(device), event_times.to(device), is_event.to(device), demo_feats.to(device), padded_matrix.to(device), list(hdf5_path_list)
        outputs = model(x_data, padded_matrix, demo_feats)
    
        logits = outputs.cpu().numpy()
        all_outputs.append(logits)
        all_event_times.append(event_times.cpu().numpy())
        all_is_event.append(is_event.cpu().numpy())
        all_paths.append(hdf5_path_list)

all_outputs = np.concatenate(all_outputs, axis=0)
all_event_times = np.concatenate(all_event_times, axis=0)
all_is_event = np.concatenate(all_is_event, axis=0)
all_paths = np.concatenate(all_paths)

outputs_path = os.path.join(save_path, "all_outputs.pickle")
event_times_path = os.path.join(save_path, "all_event_times.pickle")
is_event_path = os.path.join(save_path, "all_is_event.pickle")
file_paths = os.path.join(save_path, "all_paths.pickle")

save_data(all_outputs, outputs_path)
save_data(all_event_times, event_times_path)
save_data(all_is_event, is_event_path)
save_data(all_paths, file_paths)

In [ ]:
all_outputs.shape, all_event_times.shape, all_is_event.shape

Above, you get the model outputs, which you can then use to look for specific disease diagnosis. Nope that the shape of the output above is 1065, meaning, this model gives logprobs for 1065 conditions. We provide information about each disease index and its corresponding phecode here `sleepfm/configs/label_mapping.csv`. You can map it as follows. 

In [ ]:
labels_df = pd.read_csv("../sleepfm/configs/label_mapping.csv")

In [ ]:
labels_df["output"] = all_outputs[0]
labels_df["is_event"] = all_is_event[0]
labels_df["event_time"] = all_event_times[0]

In [ ]:
labels_df.head()

Above, you get the output hazards from our model, and also your labels for is_event and event_times. Is_event is an indicator for if the event occured and event_time is the time to event

In [ ]:
import os
import sys
import h5py
import tqdm
import random
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.metrics import f1_score, confusion_matrix

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

# =====================================================================
# 核心环境设置与包导入 (对应原 Notebook 第一个代码块)
# 注意：这里假设您的运行目录在 sleepfm-clinical/notebooks/ 下
# =====================================================================
sys.path.append("..")
sys.path.append("../sleepfm")

try:
    from preprocessing.preprocessing import EDFToHDF5Converter
    from models.dataset import SetTransformerDataset, collate_fn
    from models.models import SetTransformer, SleepEventLSTMClassifier
    from utils import load_config, load_data, save_data, count_parameters
except ImportError as e:
    print(f"导入自定义模块失败，请检查运行路径是否正确: {e}")
    sys.exit(1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"---------- 当前使用的计算设备: {device} ----------\n")


def main():
    base_save_path = "demo_data"
    os.makedirs(base_save_path, exist_ok=True)

    # =====================================================================
    # Part 0: 数据预处理 (将原始 EDF 脑电数据转换为 HDF5 格式)
    # [输出]: 会在控制台打印 "Extracting EDF parameters..." 等进度信息
    # [文件生成]: demo_data/demo_psg.hdf5
    # =====================================================================
    print("========== Part 0: 开始预处理 EDF 文件 ==========")
    root_dir = "/edf_root"      
    target_dir = "/note"    
    edf_path = "demo_data/demo_psg.edf"
    hdf5_path = os.path.join(base_save_path, "demo_psg.hdf5")

    if not os.path.exists(hdf5_path):
        converter = EDFToHDF5Converter(
            root_dir=root_dir,
            target_dir=target_dir,
            resample_rate=128
        )
        converter.convert(edf_path, hdf5_path)
    else:
        print(f"HDF5文件已存在，跳过预处理: {hdf5_path}\n")

    # =====================================================================
    # Part 1: 使用 SleepFM 基础模型生成数据特征向量 (Embeddings)
    # [输出]: 打印模型参数量 "Trainable parameters: 4.44 million"
    # [文件生成]: 在 demo_emb/ 和 demo_5min_agg_emb/ 下生成特征文件
    # =====================================================================
    print("========== Part 1: 生成特征向量 (Embeddings) ==========")
    model_path = "../sleepfm/checkpoints/model_base"
    channel_groups_path = "../sleepfm/configs/channel_groups.json"
    config_path = os.path.join(model_path, "config.json")

    config = load_config(config_path)
    channel_groups = load_data(channel_groups_path)

    # 动态加载并实例化基础模型
    model_class = getattr(sys.modules[__name__], config['model'])
    model = model_class(
        in_channels=config["in_channels"], 
        patch_size=config["patch_size"], 
        embed_dim=config["embed_dim"], 
        num_heads=config["num_heads"], 
        num_layers=config["num_layers"], 
        pooling_head=config["pooling_head"], 
        dropout=0.0
    )
    
    if device.type == "cuda":
        model = torch.nn.DataParallel(model)
    model.to(device)

    # 加载预训练权重
    checkpoint = torch.load(os.path.join(model_path, "best.pt"), map_location=device)
    model.load_state_dict(checkpoint["state_dict"])
    model.eval()

    # 加载数据
    hdf5_paths = [hdf5_path]
    dataset = SetTransformerDataset(config, channel_groups, hdf5_paths=hdf5_paths, split="test")
    dataloader = DataLoader(dataset, batch_size=16, num_workers=1, shuffle=False, collate_fn=collate_fn)

    output_dir = os.path.join(base_save_path, "demo_emb")
    output_5min_agg = os.path.join(base_save_path, "demo_5min_agg_emb")
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(output_5min_agg, exist_ok=True)

    # 模型推理：提取特征
    # 注意：为了代码简洁，去除了此处的长篇 h5py 写入逻辑，
    # 如果您的目录下已经有 demo_emb/demo_psg.hdf5，这一步实际上可以跳过。
    # (原 Notebook 中此处有生成 Embedding 的循环，若需重新生成，请将原块代码粘贴至此)
    print("特征向量生成完毕 (或已从缓存读取)\n")


    # =====================================================================
    # Part 2: 睡眠分期 (Sleep Staging) - [您的核心需求]
    # [输出]: 打印睡眠分期模型的层数、参数量
    # =====================================================================
    print("========== Part 2: 执行睡眠分期任务 ==========")
    sleep_staging_model_path = "../sleepfm/checkpoints/model_sleep_staging"
    sleep_staging_config = load_data(os.path.join(sleep_staging_model_path, "config.json"))

    sleep_staging_model_params = sleep_staging_config['model_params']
    sleep_staging_model_class = getattr(sys.modules[__name__], sleep_staging_config['model'])

    sleep_staging_model = sleep_staging_model_class(**sleep_staging_model_params).to(device)
    if device.type == "cuda":
        sleep_staging_model = nn.DataParallel(sleep_staging_model)

    print(f"Model initialized: {type(sleep_staging_model).__name__}")
    
    # 加载睡眠分期权重
    sleep_staging_checkpoint_path = os.path.join(sleep_staging_model_path, "best.pth")
    sleep_staging_checkpoint = torch.load(sleep_staging_checkpoint_path, map_location=device)
    sleep_staging_model.load_state_dict(sleep_staging_checkpoint)
    sleep_staging_model.eval()

    # 定义测试集 DataLoader (使用前面生成的 Embedding 和 CSV 标签)
    test_hdf5_paths = [os.path.join(base_save_path, "demo_emb/demo_psg.hdf5")]
    test_label_files = [os.path.join(base_save_path, "demo_psg.csv")]
    
    test_dataset = SleepEventClassificationDataset(
        sleep_staging_config, channel_groups, split="test", 
        hdf5_paths=test_hdf5_paths, label_files=test_label_files
    )
    test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, num_workers=1, collate_fn=sleep_event_finetune_full_collate_fn)

    # ---------------- 核心评估循环 ----------------
    all_targets, all_logits, all_outputs, all_masks, all_paths = [], [], [], [], []

    with torch.no_grad():
        for (x_data, y_data, padded_matrix, hdf5_path_list) in tqdm.tqdm(test_loader, desc="Evaluating"):
            x_data = x_data.to(device)
            y_data = y_data.to(device)
            padded_matrix = padded_matrix.to(device)
            
            outputs, mask = sleep_staging_model(x_data, padded_matrix)
            
            all_targets.append(y_data.cpu().numpy())
            all_outputs.append(torch.softmax(outputs, dim=-1).cpu().numpy())
            all_logits.append(outputs.cpu().numpy())
            all_masks.append(mask.cpu().numpy())
            all_paths.append(list(hdf5_path_list))

    # [文件生成]: 保存预测的分类概率、原始标签等结果用于后续分析
    save_path = os.path.join(base_save_path, "demo_sleep_staging")
    os.makedirs(save_path, exist_ok=True)
    save_data(all_targets, os.path.join(save_path, "all_targets.pickle"))
    save_data(all_outputs, os.path.join(save_path, "all_outputs.pickle"))
    save_data(all_logits, os.path.join(save_path, "all_logits.pickle"))
    save_data(all_masks, os.path.join(save_path, "all_masks.pickle"))

    # ---------------- 数据展平与过滤 ----------------
    # 将批次数据展平成一维，过滤掉被 Padding (填充) 的无用数据
    all_logits_flat = np.concatenate([logits.reshape(-1, logits.shape[-1]) for logits in all_logits], axis=0)
    all_outputs_flat = np.concatenate([outputs.reshape(-1, outputs.shape[-1]) for outputs in all_outputs], axis=0)
    all_targets_flat = np.concatenate([targets.reshape(-1) for targets in all_targets], axis=0)
    all_masks_flat = np.concatenate([mask.reshape(-1) for mask in all_masks], axis=0)

    mask_filter = all_masks_flat == 0
    all_outputs_filtered = all_outputs_flat[mask_filter]
    all_targets_filtered = all_targets_flat[mask_filter]

    # =====================================================================
    # 睡眠分期最终输出报告 (控制台输出 + 图片显示)
    # [核心输出]: F1指标 和 混淆矩阵热力图
    # =====================================================================
    print("\n========== 最终睡眠分期评估报告 ==========")
    class_labels = ["Wake", "Stage 1", "Stage 2", "Stage 3", "REM"]
    predicted_labels = np.argmax(all_outputs_filtered, axis=1)

    # 1. 输出各个睡眠阶段的 F1 Score (控制台)
    f1_scores = f1_score(all_targets_filtered, predicted_labels, average=None, labels=range(len(class_labels)))
    for idx, label in enumerate(class_labels):
        print(f"F1 Score for {label}: {f1_scores[idx]:.3f}")

    # 2. 绘制混淆矩阵 (弹窗显示图片)
    conf_matrix = confusion_matrix(all_targets_filtered, predicted_labels, labels=range(len(class_labels)))
    conf_matrix_percent = conf_matrix / conf_matrix.sum(axis=1, keepdims=True) * 100

    plt.figure(figsize=(8, 6))
    sns.heatmap(
        conf_matrix_percent,
        annot=True,
        fmt=".1f",
        cmap="Blues",
        xticklabels=class_labels,
        yticklabels=class_labels,
        annot_kws={"size": 12}, 
        cbar_kws={"shrink": 1}, 
    )
    plt.title("Sleep Staging Confusion Matrix (%)", fontsize=14)
    plt.xlabel("Predicted Labels", fontsize=12)
    plt.ylabel("True Labels", fontsize=12)
    plt.xticks(fontsize=12, ha="center") 
    plt.yticks(fontsize=12, rotation=0) 
    plt.tight_layout()
    plt.show() # 此处会弹出一个图表窗口

# =====================================================================
# 附：原 Notebook 中定义的 Dataset 与 DataLoader 处理函数
# 由于长篇定义，放在脚本底部以免影响核心阅读逻辑
# =====================================================================
class SleepEventClassificationDataset(Dataset):
    def __init__(self, config, channel_groups, hdf5_paths, label_files, split="train"):
        self.config = config
        self.max_channels = self.config["max_channels"]
        self.context = int(self.config["context"])
        self.channel_like = self.config["channel_like"]
        self.max_seq_len = config["model_params"]["max_seq_length"]
        labels_dict = {os.path.basename(p).rsplit(".", 1)[0]: p for p in label_files if os.path.exists(p)}
        hdf5_paths = [p for p in hdf5_paths if os.path.exists(p)]
        hdf5_paths = [p for p in hdf5_paths if os.path.basename(p).rsplit(".", 1)[0] in labels_dict]
        self.hdf5_paths = hdf5_paths
        self.labels_dict = labels_dict
        self.index_map = [(p, labels_dict[os.path.basename(p).rsplit(".", 1)[0]], -1) for p in self.hdf5_paths]
        self.total_len = len(self.index_map)

    def __len__(self):
        return self.total_len

    def __getitem__(self, idx):
        hdf5_path, label_path, start_index = self.index_map[idx]
        labels_df = pd.read_csv(label_path)
        labels_df["StageNumber"] = labels_df["StageNumber"].replace(-1, 0)
        y_data = labels_df["StageNumber"].to_numpy()
        x_data = []
        with h5py.File(hdf5_path, "r") as hf:
            dset_names = list(hf.keys())
            for dataset_name in dset_names:
                if dataset_name in self.channel_like:
                    x_data.append(hf[dataset_name][:])
        x_data = np.array(x_data)
        x_data = torch.tensor(x_data, dtype=torch.float32)
        y_data = torch.tensor(y_data, dtype=torch.float32)
        min_length = min(x_data.shape[1], len(y_data))
        x_data = x_data[:, :min_length, :]
        y_data = y_data[:min_length]
        return x_data, y_data, self.max_channels, self.max_seq_len, hdf5_path

def sleep_event_finetune_full_collate_fn(batch):
    x_data, y_data, max_channels_list, max_seq_len_list, hdf5_path_list = zip(*batch)
    num_channels = max(max_channels_list)
    max_seq_len_temp = max([item.size(1) for item in x_data])
    max_seq_len = max_seq_len_temp if max_seq_len_list[0] is None else min(max_seq_len_temp, max_seq_len_list[0])
    padded_x_data, padded_y_data, padded_mask = [], [], []

    for x_item, y_item in zip(x_data, y_data):
        tgt_sleep_no_sleep = np.where(y_item > 0, 1, 0)
        moving_avg_tgt_sleep_no_sleep = np.convolve(tgt_sleep_no_sleep, np.ones(1080)/1080, mode='valid')
        try:
            first_non_zero_index = np.where(moving_avg_tgt_sleep_no_sleep > 0.5)[0][0]
        except IndexError:
            first_non_zero_index = 0
        
        c, s, e = x_item.size()
        c, s = min(c, num_channels), min(s, max_seq_len + first_non_zero_index)
        
        padded_x_item = torch.zeros((num_channels, max_seq_len, e))
        mask = torch.ones((num_channels, max_seq_len))
        padded_x_item[:c, :s-first_non_zero_index, :e] = x_item[:c, first_non_zero_index:s, :e]
        mask[:c, :s-first_non_zero_index] = 0
        
        padded_y_item = torch.zeros(max_seq_len)
        padded_y_item[:s-first_non_zero_index] = y_item[first_non_zero_index:s]
        
        padded_x_data.append(padded_x_item)
        padded_y_data.append(padded_y_item)
        padded_mask.append(mask)

    return torch.stack(padded_x_data), torch.stack(padded_y_data), torch.stack(padded_mask), hdf5_path_list

if __name__ == "__main__":
    main()

In [8]:
import os
import sys
import h5py
import tqdm
import torch
import numpy as np
import pandas as pd
from torch import nn
from torch.utils.data import Dataset, DataLoader

# =====================================================================
# 核心环境设置与包导入
# =====================================================================
sys.path.append("..")
sys.path.append("../sleepfm")

from preprocessing.preprocessing import EDFToHDF5Converter
from models.dataset import SetTransformerDataset, collate_fn
from utils import load_config, load_data, count_parameters

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =====================================================================
# 【这里填写你的文件信息】
# =====================================================================
YOUR_EDF_FILENAME = "demo_psg.edf"  # <--- 您放进 demo_data 目录的 edf 文件名

base_save_path = "demo_data"
file_prefix = YOUR_EDF_FILENAME.replace(".edf", "")
edf_path = os.path.join(base_save_path, YOUR_EDF_FILENAME)
hdf5_path = os.path.join(base_save_path, f"{file_prefix}.hdf5")


def main():
    # ---------------------------------------------------------
    # Part 0: 数据预处理 (将你的 EDF 转换为模型的 HDF5)
    # ---------------------------------------------------------
    print(f"\n========== Part 0: 处理文件 {YOUR_EDF_FILENAME} ==========")
    if not os.path.exists(edf_path):
        print(f"错误：找不到文件 {edf_path}，请确认文件是否已放入该目录！")
        return

    if not os.path.exists(hdf5_path):
        converter = EDFToHDF5Converter(root_dir="/edf_root", target_dir="/note", resample_rate=128)
        converter.convert(edf_path, hdf5_path)
    else:
        print("HDF5文件已存在，跳过格式转换。\n")

    # ---------------------------------------------------------
    # Part 1: 生成特征向量 (Embeddings)
    # ---------------------------------------------------------
    print("========== Part 1: 生成并保存特征向量 ==========")
    model_path = "../sleepfm/checkpoints/model_base"
    config = load_config(os.path.join(model_path, "config.json"))
    channel_groups = load_data("../sleepfm/configs/channel_groups.json")

    # [修复点1] 直接导入基础模型类
    from models.models import SetTransformer 
    model = SetTransformer(
        in_channels=config["in_channels"], patch_size=config["patch_size"], 
        embed_dim=config["embed_dim"], num_heads=config["num_heads"], 
        num_layers=config["num_layers"], pooling_head=config["pooling_head"], dropout=0.0
    )
    if device.type == "cuda":
        model = torch.nn.DataParallel(model)
    model.to(device)
    model.load_state_dict(torch.load(os.path.join(model_path, "best.pt"), map_location=device)["state_dict"])
    model.eval()

    dataset = SetTransformerDataset(config, channel_groups, hdf5_paths=[hdf5_path], split="test")
    dataloader = DataLoader(dataset, batch_size=16, num_workers=1, shuffle=False, collate_fn=collate_fn)
    
    # [补充漏掉的核心逻辑] 将提取的特征真正写入磁盘供 Part 2 使用
    output_dir = os.path.join(base_save_path, "demo_emb")
    os.makedirs(output_dir, exist_ok=True)
    target_emb_file = os.path.join(output_dir, f"{file_prefix}.hdf5")

    # 如果还没生成过特征，就跑一遍生成循环
    if not os.path.exists(target_emb_file):
        with torch.no_grad():
            for batch in tqdm.tqdm(dataloader, desc="提取脑电特征中"):
                batch_data, mask_list, file_paths, dset_names_list, chunk_starts = batch
                (bas, resp, ekg, emg) = batch_data
                (mask_bas, mask_resp, mask_ekg, mask_emg) = mask_list

                bas, resp = bas.to(device, dtype=torch.float), resp.to(device, dtype=torch.float)
                ekg, emg = ekg.to(device, dtype=torch.float), emg.to(device, dtype=torch.float)
                mask_bas, mask_resp = mask_bas.to(device, dtype=torch.bool), mask_resp.to(device, dtype=torch.bool)
                mask_ekg, mask_emg = mask_ekg.to(device, dtype=torch.bool), mask_emg.to(device, dtype=torch.bool)

                embeddings = [
                    model(bas, mask_bas), model(resp, mask_resp),
                    model(ekg, mask_ekg), model(emg, mask_emg),
                ]

                # 取出颗粒度为5秒的特征 (e[1])
                embeddings_new = [e[1] for e in embeddings]

                for i in range(len(file_paths)):
                    file_path = file_paths[i]
                    chunk_start = chunk_starts[i]
                    subject_id = os.path.basename(file_path).split('.')[0]
                    output_path = os.path.join(output_dir, f"{subject_id}.hdf5")

                    with h5py.File(output_path, 'a') as hdf5_file:
                        for modality_idx, modality_type in enumerate(config["modality_types"]):
                            if modality_type in hdf5_file:
                                dset = hdf5_file[modality_type]
                                chunk_start_correct = chunk_start // (config["embed_dim"] * 5)
                                chunk_end = chunk_start_correct + embeddings_new[modality_idx][i].shape[0]
                                if dset.shape[0] < chunk_end:
                                    dset.resize((chunk_end,) + embeddings_new[modality_idx][i].shape[1:])
                                dset[chunk_start_correct:chunk_end] = embeddings_new[modality_idx][i].cpu().numpy()
                            else:
                                hdf5_file.create_dataset(
                                    modality_type, 
                                    data=embeddings_new[modality_idx][i].cpu().numpy(), 
                                    chunks=(config["embed_dim"],) + embeddings_new[modality_idx][i].shape[1:], 
                                    maxshape=(None,) + embeddings_new[modality_idx][i].shape[1:]
                                )
        print("✅ 新文件特征提取并保存完毕！")
    else:
        print(f"✅ 发现已缓存的特征文件：{target_emb_file}，跳过提取。")

    # ---------------------------------------------------------
    # Part 2: 睡眠分期推理 (不依赖 CSV 标签文件)
    # ---------------------------------------------------------
    print("\n========== Part 2: 预测睡眠分期 ==========")
    sleep_staging_model_path = "../sleepfm/checkpoints/model_sleep_staging"
    sleep_staging_config = load_data(os.path.join(sleep_staging_model_path, "config.json"))

    # [修复点2] 解决报错：直接导入分期模型类
    from models.models import SleepEventLSTMClassifier
    sleep_staging_model = SleepEventLSTMClassifier(**sleep_staging_config['model_params']).to(device)
    
    if device.type == "cuda":
        sleep_staging_model = nn.DataParallel(sleep_staging_model)

    sleep_staging_model.load_state_dict(torch.load(os.path.join(sleep_staging_model_path, "best.pth"), map_location=device))
    sleep_staging_model.eval()

    test_hdf5_paths = [target_emb_file]
    test_dataset = NoLabelSleepDataset(sleep_staging_config, channel_groups, hdf5_paths=test_hdf5_paths)
    
    if len(test_dataset) == 0:
         print("❌ 错误：Dataset 为空，请检查 Part 1 特征是否成功生成。")
         return

    test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, num_workers=1, collate_fn=sleep_event_finetune_full_collate_fn)

    all_outputs, all_masks = [], []
    with torch.no_grad():
        for (x_data, y_data, padded_matrix, hdf5_path_list) in tqdm.tqdm(test_loader, desc="模型推理中"):
            x_data = x_data.to(device)
            padded_matrix = padded_matrix.to(device)
            outputs, mask = sleep_staging_model(x_data, padded_matrix)
            all_outputs.append(torch.softmax(outputs, dim=-1).cpu().numpy())
            all_masks.append(mask.cpu().numpy())

    all_outputs_flat = np.concatenate([out.reshape(-1, out.shape[-1]) for out in all_outputs], axis=0)
    all_masks_flat = np.concatenate([m.reshape(-1) for m in all_masks], axis=0)
    mask_filter = all_masks_flat == 0
    valid_outputs = all_outputs_flat[mask_filter]

    predicted_labels = np.argmax(valid_outputs, axis=1)

    # ---------------------------------------------------------
    # 导出结果到 CSV 供您查看
    # ---------------------------------------------------------
    class_names = {0: "Wake(清醒)", 1: "Stage 1(浅睡1)", 2: "Stage 2(浅睡2)", 3: "Stage 3(深睡)", 4: "REM(做梦)"}
    predicted_names = [class_names[idx] for idx in predicted_labels]

    output_csv = os.path.join(base_save_path, f"{file_prefix}_预测结果.csv")
    df_preds = pd.DataFrame({
        "时间段(Epoch)": range(1, len(predicted_labels) + 1),
        "分期阶段代码": predicted_labels,
        "对应睡眠状态": predicted_names
    })
    df_preds.to_csv(output_csv, index=False, encoding='utf-8-sig')
    print(f"\n🎉 预测完成！")
    print(f"您的睡眠分期数据已成功导出为表格，请在此处查看: {output_csv}")


# =====================================================================
# 改造的 Dataset
# =====================================================================
class NoLabelSleepDataset(Dataset):
    def __init__(self, config, channel_groups, hdf5_paths):
        self.config = config
        self.max_channels = self.config["max_channels"]
        self.channel_like = self.config["channel_like"]
        self.max_seq_len = config["model_params"]["max_seq_length"]
        self.hdf5_paths = [p for p in hdf5_paths if os.path.exists(p)]
        self.total_len = len(self.hdf5_paths)

    def __len__(self):
        return self.total_len

    def __getitem__(self, idx):
        hdf5_path = self.hdf5_paths[idx]
        x_data = []
        with h5py.File(hdf5_path, "r") as hf:
            for dataset_name in list(hf.keys()):
                if dataset_name in self.channel_like:
                    x_data.append(hf[dataset_name][:])
                    
        x_data = torch.tensor(np.array(x_data), dtype=torch.float32)
        seq_len = x_data.shape[1]
        y_data = torch.zeros(seq_len, dtype=torch.float32)
        
        min_length = min(x_data.shape[1], len(y_data))
        return x_data[:, :min_length, :], y_data[:min_length], self.max_channels, self.max_seq_len, hdf5_path

def sleep_event_finetune_full_collate_fn(batch):
    x_data, y_data, max_channels_list, max_seq_len_list, hdf5_path_list = zip(*batch)
    num_channels = max(max_channels_list)
    max_seq_len = max([item.size(1) for item in x_data]) if max_seq_len_list[0] is None else min(max([item.size(1) for item in x_data]), max_seq_len_list[0])
    
    padded_x_data, padded_y_data, padded_mask = [], [], []
    for x_item, y_item in zip(x_data, y_data):
        c, s, e = x_item.size()
        c, s = min(c, num_channels), min(s, max_seq_len)
        padded_x_item = torch.zeros((num_channels, max_seq_len, e))
        mask = torch.ones((num_channels, max_seq_len))
        padded_x_item[:c, :s, :e] = x_item[:c, :s, :e]
        mask[:c, :s] = 0
        padded_y_item = torch.zeros(max_seq_len)
        padded_y_item[:s] = y_item[:s]
        
        padded_x_data.append(padded_x_item)
        padded_y_data.append(padded_y_item)
        padded_mask.append(mask)

    return torch.stack(padded_x_data), torch.stack(padded_y_data), torch.stack(padded_mask), hdf5_path_list

if __name__ == "__main__":
    main()


========== Part 0: 处理文件 demo_psg.edf ==========
HDF5文件已存在，跳过格式转换。

========== Part 1: 生成并保存特征向量 ==========


Indexing files: 100%|██████████| 1/1 [00:00<00:00, 270.43it/s]


✅ 发现已缓存的特征文件：demo_data/demo_emb/demo_psg.hdf5，跳过提取。

========== Part 2: 预测睡眠分期 ==========


模型推理中: 100%|██████████| 1/1 [00:00<00:00,  1.44it/s]


🎉 预测完成！
您的睡眠分期数据已成功导出为表格，请在此处查看: demo_data/demo_psg_预测结果.csv
